# TuneGen — Transformer Training
This notebook trains the TuneGen Transformer model on the GiantMIDI preprocessed dataset.

**Before running:**
1. Set Runtime → Change runtime type → T4 GPU
2. Make sure `train.npz`, `val.npz`, `test.npz` are uploaded to your Google Drive under `MyDrive/TuneGen/data/`
3. Run cells in order

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Verify GPU

In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found — check Runtime > Change runtime type > T4 GPU')

Device : cuda
GPU    : Tesla T4
Memory : 15.6 GB


## 3. Copy data from Drive to Colab local storage
Copying to local storage is faster than reading directly from Drive during training.

In [3]:
import shutil
from pathlib import Path

DRIVE_DATA_DIR  = Path('/content/drive/MyDrive/TuneGen/data')
LOCAL_DATA_DIR  = Path('/content/data')
LOCAL_DATA_DIR.mkdir(exist_ok=True)

for filename in ['train.npz', 'val.npz', 'test.npz']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if not dst.exists():
        print(f'Copying {filename}...')
        shutil.copy(src, dst)
        print(f'  Done — {dst.stat().st_size / 1e9:.2f} GB')
    else:
        print(f'{filename} already exists locally.')

train.npz already exists locally.
val.npz already exists locally.
test.npz already exists locally.


## 4. Upload model.py and train.py
Upload both files from your local machine using the cell below.

In [4]:
from google.colab import files
uploaded = files.upload()  # upload model.py and train.py when prompted

## 5. Update paths for Colab
Patches `train.py` to point to Colab local storage instead of the local `data/` folder.

In [5]:
with open('train.py', 'r') as f:
    content = f.read()

content = content.replace(
    'DATA_DIR         = Path("data")',
    'DATA_DIR         = Path("/content/data")'
).replace(
    'CHECKPOINT_DIR   = Path("checkpoints")',
    'CHECKPOINT_DIR   = Path("/content/drive/MyDrive/TuneGen/checkpoints")'
)

with open('train.py', 'w') as f:
    f.write(content)

print('Paths updated.')

Paths updated.


## 6. Update device in train.py for CUDA

In [6]:
with open('train.py', 'r') as f:
    content = f.read()

content = content.replace(
    'device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")',
    'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")'
)

with open('train.py', 'w') as f:
    f.write(content)

print('Device updated to CUDA.')

Device updated to CUDA.


## 7. Start training

In [7]:
%run train.py


  Device : cuda

  Loading data...
  Sampled 6,131,764 / 30,658,824 pairs (20%)
  Train batches : 11,977
  Val batches   : 7,486
  Parameters    : 3,358,848

  Starting training for 3 epochs...


  Epoch 1/3 complete
  Train loss : 3.0961
  Val loss   : 2.9072
  Time       : 1940.6s

  Checkpoint saved (val loss: 2.9072)


  Epoch 2/3 complete
  Train loss : 2.9329
  Val loss   : 2.8447
  Time       : 1939.5s

  Checkpoint saved (val loss: 2.8447)


  Epoch 3/3 complete
  Train loss : 2.8808
  Val loss   : 2.8150
  Time       : 1942.3s

  Checkpoint saved (val loss: 2.8150)

  Training complete. Best val loss: 2.8150
  Log saved to /content/drive/MyDrive/TuneGen/checkpoints/training_log.txt



## 8. Verify checkpoint saved to Drive

In [8]:
checkpoint_path = Path('/content/drive/MyDrive/TuneGen/checkpoints/best_model.pt')

if checkpoint_path.exists():
    size_mb = checkpoint_path.stat().st_size / 1e6
    print(f'Checkpoint found: {checkpoint_path}')
    print(f'Size: {size_mb:.1f} MB')
else:
    print('No checkpoint found — training may not have completed.')

Checkpoint found: /content/drive/MyDrive/TuneGen/checkpoints/best_model.pt
Size: 40.3 MB
